# Práctica 2

1. Módulo de Retrieval Denso
• Construcción del Índice: Utilizar el conjunto de entrenamiento de P1 (Xtrain y ytrain) como corpus.
• Función de Embedding: Definir embed_fn(texts) usando el encoder del modelo Transformer de P1 (sin la capa de clasificación) para obtener vectores d-dimensionales.
• Índice k-NN: Construir un índice (por ejemplo, con sklearn.NearestNeighbors y métrica coseno).
• Búsqueda: Definir search(query_texts, k) para devolver los k vecinos más cercanos, la distancia/similitud, y metadatos (texto original, etiqueta).
• Evaluación Básica: Calcular Precision@k y/o Recall@k de los vecinos que tienen la misma etiqueta que el ejemplo objetivo.
2. Clasificador k-NN sobre Embeddings
• Definición del Clasificador: Implementar un clasificador que obtenga el embedding del texto, recupere los top−k vecinos de entrenamiento y prediga la clase mediante voto mayoritario de las etiquetas de los vecinos.
• Evaluación: Medir el rendimiento (Accuracy y Macro-F1) usando la misma partición de datos (train/dev/test) que en P1.
• Comparación: Comparar el rendimiento del k-NN con el modelo clásico (TF-IDF + clasificador de P1) y el Transformer fine-tuneado de P1.
3. Clasificador Híbrido (RAG para Clasificación)
• Diseño Híbrido: Combinar las probabilidades del Transformer de P1 (p 
T
​
 ) y la distribución de clases en los vecinos (p 
K
​
 ).
• Fórmula: Implementar la combinación utilizando p 
comb
​
 (y∣x)=αp 
T
​
 (y∣x)+(1−α)p 
K
​
 (y∣x).
• Experimentación: Evaluar para varios valores de α (e.g., 0.0, 0.25, 0.5, 0.75, 1.0) y un valor de k fijo.
• Evaluación: Reportar Accuracy y Macro-F1 en el conjunto de test para los distintos α.
4. Explicabilidad basada en Retrieval
• Selección de Ejemplos: Elegir al menos 10 ejemplos de test bien clasificados y 10 mal clasificados por el modelo híbrido.
• Visualización: Para cada ejemplo, mostrar el texto original, las predicciones del híbrido y del Transformer, y la lista de los k vecinos recuperados (texto y etiqueta).
• Comentario Explicativo: Redactar un comentario breve (2–3 frases) explicando por qué el modelo eligió la clase predicha basándose en las expresiones presentes en los vecinos similares.
5. Compresión / Modelo Destilado
• Elección del Modelo: Seleccionar un modelo comprimido/destilado (como DistilBERT, MiniLM, o ALBERT) que sea notablemente más ligero que el Transformer de P1.
• Entrenamiento: Ajustar el modelo en el mismo dataset y split train/dev/test que en P1.
• Comparativa de Calidad: Evaluar y reportar Accuracy, Macro-F1 y F1 por clase.
• Comparativa de Coste: Reportar el número aproximado de parámetros, el tamaño en disco (MB) y el tiempo medio de inferencia.
6. Resumen Automático como Explicación
• Resumen Global por Clase: Para cada clase, seleccionar ≈50 textos de entrenamiento, concatenar fragmentos en un "pseudo-documento" y usar un modelo generativo (p. ej., T5-small o BART-base) para generar un resumen de 3–5 frases que describa la clase.
• Resumen Local (Explicación de Predicciones): Seleccionar al menos 6 ejemplos de test (3 correctos, 3 erróneos).
• Generación de Explicación: Usar el módulo de retrieval para obtener los k vecinos, construir un texto de entrada que incluya el texto objetivo y los vecinos, y generar un resumen de 2–3 frases explicando el acierto/error basado en esos vecinos.
7. Análisis Global y Discusión (A incluir en el Informe)
• Comparación Global: Comparar los baselines de P1, k-NN, el híbrido y el modelo comprimido, analizando el equilibrio entre calidad, coste e interpretabilidad.
• Sesgo y Ética: Analizar aspectos de equidad y sesgo, como si hay clases peor tratadas o si los vecinos podrían agravar sesgos.
• Utilidad de Explicabilidad: Reflexionar sobre la utilidad real de las explicaciones basadas en tokens/atributos (P1), ejemplos (retrieval) y explicaciones generativas (resumen).
Tarea Opcional (Para Matrícula de Honor, MH)
• Knowledge Distillation: Definir un modelo student más pequeño y entrenarlo utilizando la función de pérdida sobre las etiquetas reales más la divergencia KL entre las distribuciones del teacher (P1 Transformer) y el student (con temperatura).

## Preparación del Entorno y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch

# Importar módulos propios
from src.models.retrieval import DenseRetriever

# Configurar el path a los datos (ajustar si es necesario)
data_path = Path("../../practica1/pln2_practica1/data/en_song_lyrics_clear.csv")
if not data_path.exists():
    # Fallback to local if copied
    data_path = Path("data/en_song_lyrics_clear.csv")

print(f"Cargando datos desde {data_path}...")
try:
    df = pd.read_csv(data_path)
    # Sample for quick dev if needed (uncomment to test speed)
    # df = df.groupby('label').sample(n=100, random_state=42)
except FileNotFoundError:
    print("ERROR: No se encontró el fichero de datos. Verifica la ruta.")
    # Create dummy data for structure if loading fails
    df = pd.DataFrame({'text': ['sample text'] * 100, 'label': ['Pop'] * 100})

# Split estratificado como en P1
X = df['text']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

c:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\PLN2\Practicas\practica2\pln2_practica2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando datos desde ..\..\practica1\pln2_practica1\data\en_song_lyrics_clear.csv...
Train: 360868, Val: 77329, Test: 77329


: 

## 1. Módulo de Retrieval Denso

In [ ]:
# Constuir el retriever
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'roberta-base' # Usar el mismo base que P1

retriever = DenseRetriever(model_name_or_path=model_name, device=device)

# Construir índice con Train
retriever.build_index(X_train.tolist(), y_train.tolist(), n_neighbors=5)

Loading model from roberta-base...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Building index with 360868 documents...


Embedding:  27%|██▋       | 3047/11278 [2:37:48<6:47:42,  2.97s/it] 

In [ ]:
# Evaluación Básica (Precision@k, Recall@k)
def evaluate_retrieval(retriever, texts, labels, k=5):
    results, _, _ = retriever.search(texts, k=k)
    precisions = []
    # Recall es confuso en k-NN puro si no sabemos cuantos relevantes totales hay, 
    # aquí asumimos que 'relevantes' son los de la misma clase.
    
    for i, res in enumerate(results):
        target_label = labels[i]
        relevant_matches = sum(1 for item in res if item['label'] == target_label)
        precisions.append(relevant_matches / k)
    return np.mean(precisions)

# Evaluar en una muestra de Test (para rapidez)
test_subset_size = min(200, len(X_test))
indices_test = np.random.choice(len(X_test), test_subset_size, replace=False)
X_test_sample = X_test.iloc[indices_test].tolist()
y_test_sample = y_test.iloc[indices_test].tolist()

p_at_k = evaluate_retrieval(retriever, X_test_sample, y_test_sample, k=5)
print(f"Precision@5 (en muestra de {test_subset_size}): {p_at_k:.4f}")

## 2. Clasificador k-NN sobre Embeddings

In [ ]:
from collections import Counter

class KNNClassifier:
    def __init__(self, retriever, k=5):
        self.retriever = retriever
        self.k = k
        
    def predict(self, texts):
        results, _, _ = self.retriever.search(texts, k=self.k)
        preds = []
        for res in results:
            labels = [item['label'] for item in res]
            # Voto mayoritario
            most_common = Counter(labels).most_common(1)[0][0]
            preds.append(most_common)
        return preds

knn_clf = KNNClassifier(retriever, k=5)

# Evaluación en Test
print("Evaluando Clasificador k-NN...")
# Usamos la muestra por eficiencia, pero para el reporte final usar todo X_test
y_pred_knn = knn_clf.predict(X_test_sample)

print("Accuracy:", accuracy_score(y_test_sample, y_pred_knn))
print("Macro F1:", f1_score(y_test_sample, y_pred_knn, average='macro'))

## 3. Clasificador Híbrido (RAG)

In [ ]:
# Necesitamos las probabilidades del Transformer (P_T) y las del k-NN (P_K)
# P_K se puede estimar como frecuencia_clase / k

def get_knn_probs(retriever, texts, k, all_labels):
    results, _, _ = retriever.search(texts, k=k)
    probs_list = []
    label_to_idx = {l: i for i, l in enumerate(all_labels)}
    
    for res in results:
        counts = Counter([item['label'] for item in res])
        probs = np.zeros(len(all_labels))
        for label, count in counts.items():
            if label in label_to_idx:
                probs[label_to_idx[label]] = count / k
        probs_list.append(probs)
    return np.array(probs_list)

# Simulación de Transformer Probs (P_T)
# En la práctica real, usaríamos: model(inputs).logits.softmax(dim=-1)
# Aquí usaremos un placeholder o cargaremos el modelo completo si es posible

def get_transformer_probs(model, tokenizer, texts, device):
    # TODO: Implementar inferencia real con el modelo de clasificación
    # Esto requiere que el modelo cargado en retriever sea AutoModelForSequenceClassification
    # Si retriever.model es solo base, necesitamos cargar la cabecera o el modelo completo aparte.
    
    # Supongamos que cargamos el modelo de clasificacion:
    pass 
    return np.random.rand(len(texts), 6) # Placeholder

# Implementación del Híbrido
unique_labels = sorted(list(set(y_train)))
print("Etiquetas:", unique_labels)

# Para ejecutar esto se necesita el modelo de clasificacion entrenado.
